In [1]:
import numpy as np
import torch
import hydra
from omegaconf import DictConfig

from omniisaacgymenvs.utils.hydra_cfg.hydra_utils import *
from omniisaacgymenvs.utils.hydra_cfg.reformat import omegaconf_to_dict, print_dict

from omniisaacgymenvs.utils.task_util import initialize_task
from omniisaacgymenvs.envs.vec_env_rlgames import VecEnvRLGames

In [16]:
from os.path import join, dirname
import os
import sys

from einops import rearrange, repeat

sys.path.append(join(dirname(dirname(os.getcwd()))))

from configuration import cf

from memory_perceiver.core.model_net import PerceiverIO
from memory_perceiver.processing.vectorize import input_to_tensor, fourier_encode

%load_ext autoreload
%autoreload 2
%env ISAAC_JUPYTER_KERNEL=1

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: ISAAC_JUPYTER_KERNEL=1


In [5]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [6]:
nb_frames = 32
frames_future = 8
len_video_frame = 1824  # 456*256/(8*8)
len_audio_frame = 120   # 1920/16
query_dim = 832
video_latent_dim = 256
audio_latent_dim = 560
num_workers = 2
batch_size = 4

len_vid = nb_frames*len_video_frame
len_audio = nb_frames*len_audio_frame
len_audio_raw = len_audio*cf.sample_per_frame
video_future = frames_future*len_video_frame
audio_future = frames_future*len_audio_frame

special_compression = [i for i in range(video_future)] + [i for i in range(len_vid, len_vid+audio_future)]
for k in range(len(cf.body_parts)):
    special_compression += [ len_vid + len_audio + k*nb_frames + i for i in range(frames_future) ]

In [7]:
def create_model(device=device):

    # Projection to return modalities, enhancement

    modality_parameters = {}
    modality_parameters['audio'] = audio_latent_dim
    modality_parameters['video'] = video_latent_dim
    for part, params in cf.body_parts.items():
        modality_parameters[part] = query_dim - 2*128 - params

    projection_parameters = dict(cf.body_parts)
    projection_parameters.update({'audio': 16})

    # Instantiate model
        
    model = PerceiverIO(
            2,
            query_dim,
            special_compression = special_compression,
            device=device,
            projection_parameters = projection_parameters,
            modality_parameters = modality_parameters,
            max_batch_size = 4
        )
    model = model.to(device)

    # Queries

    queries = []
    audio_query = repeat( torch.cat([model.modality['audio'], torch.zeros(16, device=device)]), '... -> n ...', n=audio_future)
    audio_pos = fourier_encode(torch.linspace(-1, 1, audio_future), audio_future/2, num_bands=128).to(device)
    queries.append(torch.cat([audio_query, audio_pos], dim=-1).to(device))
    for part, params in cf.body_parts.items():
        part_query = repeat( torch.cat([model.modality[part], torch.zeros(params, device=device)]), '... -> n ...', n=frames_future)
        part_pos = fourier_encode(torch.linspace(-1, 1, frames_future), frames_future/2, num_bands=128).to(device)
        queries.append(torch.cat([part_query, part_pos], dim=-1).to(device))
    queries=torch.cat(queries)


    return model, queries

In [8]:
model, queries = create_model()

In [9]:
model.modality.keys()

dict_keys(['audio', 'video', 'observations'])

In [13]:
inputs={}
inputs['video']=torch.randn(4, 3, 32, 256, 456, device=device)
inputs['audio']=torch.randn(4, 32*1920, device=device)
inputs['observations']=torch.randn(4, 32, 4, device=device)

In [14]:
batch = input_to_tensor(inputs, model.modality, cf.body_parts).to(device)

In [17]:
with torch.no_grad():
    x, aux_loss = model(batch, queries = queries)

In [ ]:
@hydra.main(config_name="config", config_path="../cfg")
def parse_hydra_configs(cfg: DictConfig):

    cfg_dict = omegaconf_to_dict(cfg)
    print_dict(cfg_dict)

    headless = cfg.headless
    render = not headless

    env = VecEnvRLGames(headless=headless, sim_device=cfg.device_id)
    # sets seed. if seed is -1 will pick a random one
    from omni.isaac.core.utils.torch.maths import set_seed
    cfg.seed = set_seed(cfg.seed, torch_deterministic=cfg.torch_deterministic)
    cfg_dict['seed'] = cfg.seed
    task = initialize_task(cfg_dict, env)

    save_step = 160
    obs_buffer = []
    reward_buffer = []
    action_buffer = []
    while env._simulation_app.is_running():
        if env._world.is_playing():
            if env._world.current_time_step_index == 0:
                env._world.reset(soft=True)

            # 'cartpole_view' 'obs_buf' 512, 4
            obs = env._task.get_observations()
            reward = env._task.calculate_metrics()
            print('obs: ', obs, '\n', '$$$', '\n') 

            
            # obs_buffer.append(obs)
            # if(len(obs_buffer)>32):
                #  obs_buffer.pop(0)
            # reward_buffer.append(reward)
            # if(len(reward_buffer)>32):
                #  reward_buffer.pop(0)
            # media['video'] = default_video
            # media['audio'] = default_audio
            #
            # if env.sim_frame_count % save_step == 0 and len(obs_buffer) >= save_step:
            # Save mem body, audio, video


            # if(len(action_buffer) == 0 and len(obs_buffer) >= 8):
                # obs.append(get camera view())
                # obs.append(audio)
                # with torch.no_grad():
                    # action, voice = memory_perceiver(obs_buffer[-8:])
                # action_buffer.append(action)
            # if(len(action_buffer)>0):
                # action = action_buffer.pop(0)
            # else:
                # action = torch.randn(512, 1)


            action = torch.tensor(np.array([env.action_space.sample() for _ in range(env.num_envs)]), device=task.rl_device)
            # 512, 1
            print('action: ', action.size(), '\n', '$$$', '\n') 
            env._task.pre_physics_step(action)
            env._world.step(render=render)
            env.sim_frame_count += 1
            env._task.post_physics_step()
        else:
            env._world.step(render=render)

    env._simulation_app.close()

if __name__ == '__main__':
    parse_hydra_configs()
